In [ ]:
import torch
from obj.RawData import *
from obj.helpers import *
from obj.FSLViewer import FSLViewer
from obj.Smap import Smaps
import gc

In [ ]:
viewer = FSLViewer(halted=False)

device = best_available_device()
torch.manual_seed(0)
RawData.set_verbose()

smap = Smaps.load("/home/clouly/DEV/asldata/20260929/smaps/ball_smap.mat", cdim=-1).to(torch.complex64).to(device)#.compress(4)
viewer.add_smap(smap)
print(smap)

In [ ]:
y_B0_raw = [y.initial_clip().coil_compress(smap) for y in RawData.from_glob("/home/clouly/DEV/asldata/20260929/asldata_te*_a20")]
TEs = torch.tensor([3.5, 5.4, 7.4, 10.5])

In [ ]:
x_B0 = []
for i, y in enumerate(y_B0_raw): 
    y = y.flatten(1, 2).unsqueeze(2).set_mirt_shape()
    w = y.get_dcf(device=device)
    x_img, _ = y.recon_CG(smap, 1, w, device, batchmode=True)
    x_img = x_img.squeeze()
    viewer.add_complex(x_img, f"TE_{int(TEs[i].item())}")
    x_B0 += [x_img]

In [ ]:
y_B1_raw = [y.initial_clip().coil_compress(smap) for y in RawData.from_glob("/home/clouly/DEV/asldata/20260929/asldata_te03_a*")]
flips = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0, 70.0])


In [ ]:
x_B1 = []
for i, y in enumerate(y_B1_raw): 
    y = y.flatten(1, 2).unsqueeze(2).set_mirt_shape()
    w = y.get_dcf(device=device)
    x_img, _ = y.recon_CG(smap, 1, w, device, batchmode=True)
    x_img = x_img.squeeze()
    viewer.add_complex(x_img, f"flip_{int(flips[i].item())}")
    x_B1 += [x_img]

In [ ]:
viewer.close()